## 1. Modelo Transformer

Se abordó la clasificación binaria de sentimiento en español mediante BETO cased,
`dccuchile/bert-base-spanish-wwm-cased`, con su tokenizer de Hugging Face.
La revisión del modelo se fijó en la configuración para asegurar la trazabilidad
experimental. Este notebook documenta los datos, el análisis de tokenización y
la preparación del protocolo de entrenamiento.

## 2. Datos utilizados

Se reutilizaron las mismas particiones **train**, **validation** y **test** del
baseline, sin repetir la limpieza, la deduplicación ni la división del corpus.

- Predictor único: `comentario_limpio`.
- Variable objetivo: `sentimiento_id`; Negativo = 0 y Positivo = 1.
- `nombreAspecto`, `Valor` y `comentario` no se utilizaron como predictores.
- `indice_original` se conservó exclusivamente para trazabilidad.

Train se utilizó para el ajuste y validation para la selección de modelos.
Test quedó reservado para la evaluación predictiva final; las comprobaciones de
integridad y estructura no se utilizaron para tomar decisiones metodológicas.

In [1]:
from pathlib import Path
import sys

# Funciona desde la raíz o desde notebooks/, sin rutas personales.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/fase0.py").is_file()
             and (p / "configs/transformer.json").is_file()), None)
if root is None:
    raise RuntimeError("Abre el notebook dentro del repositorio con Fase 0 disponible.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import pandas as pd
from IPython.display import display
from scripts.transformer import (
    load_config, set_seed, load_verified_partitions, load_tokenizer,
    tokenizer_metadata, analyze_train_lengths, record_max_length, save_analysis,
)

config_path = root / "configs/transformer.json"
config = load_config(config_path)
seed_status = set_seed(config["seed"])
print("Modelo:", config["model_id"])
print("Semillas:", seed_status)

Modelo: dccuchile/bert-base-spanish-wwm-cased
Semillas: {'seed': 42, 'python_random': True, 'numpy': True, 'torch': False}


### Verificación de las particiones

Se comprobaron los hashes, los tamaños, las etiquetas, el esquema y la separación
entre particiones mediante la interfaz de carga común. Se conservaron las columnas
y los registros originales de cada partición. Los recuentos siguientes son
comprobaciones descriptivas de los datos, no métricas predictivas.

In [2]:
partitions = load_verified_partitions(root)
partition_summary = pd.DataFrame([
    {"partición": name, "observaciones": len(frame),
     "Negativo (0)": int(frame["sentimiento_id"].eq(0).sum()),
     "Positivo (1)": int(frame["sentimiento_id"].eq(1).sum())}
    for name, frame in partitions.items()
])
display(partition_summary)
for name, frame in partitions.items():
    print(name, "columnas:", list(frame.columns))
print("Tamaños esperados: train=16186, validation=3468, test=3469: verificados.")
print("Etiquetas {0, 1} e integridad Fase 0: verificadas.")
print("Entrada exclusiva:", config["text_column"], "| Etiqueta:", config["target_column"])
print("indice_original conservado únicamente para trazabilidad.")

,partición,observaciones,Negativo (0),Positivo (1)
0,train,16186,6870,9316
1,validation,3468,1472,1996
2,test,3469,1473,1996


train columnas: ['indice_original', 'comentario', 'Valor', 'nombreAspecto', 'comentario_limpio', 'sentimiento_id']
validation columnas: ['indice_original', 'comentario', 'Valor', 'nombreAspecto', 'comentario_limpio', 'sentimiento_id']
test columnas: ['indice_original', 'comentario', 'Valor', 'nombreAspecto', 'comentario_limpio', 'sentimiento_id']
Tamaños esperados: train=16186, validation=3468, test=3469: verificados.
Etiquetas {0, 1} e integridad Fase 0: verificadas.
Entrada exclusiva: comentario_limpio | Etiqueta: sentimiento_id
indice_original conservado únicamente para trazabilidad.


## 3. Tokenización y longitud máxima

Se cargó el tokenizer de BETO mediante `AutoTokenizer.from_pretrained`, con el
identificador y la revisión fijados en la configuración y sin ejecutar código
remoto. En esta etapa se cargaron únicamente los archivos de tokenización,
sin cargar el clasificador ni los pesos del modelo.

La clase y el tamaño del vocabulario se registraron a partir del tokenizer cargado.

In [3]:
tokenizer = load_tokenizer(config, root)
tokenizer_info = tokenizer_metadata(tokenizer, config)
display(pd.DataFrame([
    {"propiedad": key, "valor": str(value)}
    for key, value in tokenizer_info.items()
]))

/home/majovire/Escritorio/Visual Studio Code/Ciencia de Datos/Herramientas de IA/pln-sentimiento-estudiantil/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


,propiedad,valor
0,model_id,dccuchile/bert-base-spanish-wwm-cased
1,tokenizer_id,dccuchile/bert-base-spanish-wwm-cased
2,revision,c4d86612f51b4f46759c8390d1798c2febe71b93
3,class,BertTokenizerFast
4,is_fast,True
5,model_max_length,512
6,vocab_size,31002
7,vocab_with_added_tokens,31002
8,special_tokens,"{'unk_token': '[UNK]', 'sep_token': '[SEP]', '..."
9,special_token_ids,"{'[UNK]': 3, '[SEP]': 5, '[PAD]': 1, '[CLS]': ..."


### Análisis de longitud tokenizada

Se analizó exclusivamente `comentario_limpio` de **train**, sin truncamiento ni
padding e incluyendo los tokens especiales de cada secuencia. La tokenización
por lotes limitó el uso de memoria. Los percentiles se calcularon mediante
interpolación lineal de NumPy.

La regla se fijó antes de observar las longitudes: elegir el menor candidato de
`[64, 128, 256, 512]` que cubriera al menos el 99 % de train. Las secuencias de
longitud igual al límite se consideraron cubiertas. Si ningún candidato cumplía
el criterio, la regla establecía utilizar 512 e informar el porcentaje de
truncamiento. Validation y test no intervinieron en esta decisión.

In [4]:
analysis = analyze_train_lengths(tokenizer, config, root)
print("Partición analizada:", analysis["split"], "| Observaciones:", analysis["n_observations"])
stat_names = {"min": "Mínimo", "mean": "Media", "median": "Mediana",
              "p90": "P90", "p95": "P95", "p99": "P99", "max": "Máximo"}
display(pd.DataFrame([
    {"estadístico": stat_names[key], "tokens": value}
    for key, value in analysis["statistics"].items()
]))
coverage_table = pd.DataFrame(analysis["coverage"]).rename(columns={
    "candidate": "límite candidato", "exceeds_count": "cantidad que excede",
    "exceeds_percent": "% que excede", "covered_percent": "% cubierto",
})
display(coverage_table)
print("max_length seleccionado:", analysis["max_length"])
print("Cobertura de al menos el 99 %:", analysis["coverage_target_met"])
print("Porcentaje que se truncaría:", analysis["selected_truncation_percent"])
if not analysis["coverage_target_met"]:
    print("Ningún candidato alcanza el 99 %; se utiliza el máximo permitido, 512.")

Token indices sequence length is longer than the specified maximum sequence length for this model (610 > 512). Running this sequence through the model will result in indexing errors


Partición analizada: train | Observaciones: 16186


,estadístico,tokens
0,Mínimo,3.000000
1,Media,20.284011
2,Mediana,14.000000
3,P90,38.000000
4,P95,52.000000
5,P99,103.150000
6,Máximo,610.000000


,límite candidato,cantidad que excede,% que excede,% cubierto
0,64,494,3.052020,96.947980
1,128,100,0.617818,99.382182
2,256,17,0.105029,99.894971
3,512,1,0.006178,99.993822


max_length seleccionado: 128
Cobertura de al menos el 99 %: True
Porcentaje que se truncaría: 0.6178178672927221


### Registro de la longitud seleccionada

El análisis de train determinó `max_length=128`. La decisión se registró en la
configuración después de verificar el protocolo, la revisión del tokenizer y
los hashes de train. Se guardó un resumen descriptivo de longitudes y cobertura,
sin textos, predicciones ni métricas de clasificación.

In [5]:
config = record_max_length(analysis, config_path, root)
analysis_path = save_analysis(analysis, root / "results/transformer/token_length_analysis.json")
print("Decisión registrada explícitamente:", config["max_length"])
print("Resumen descriptivo:", analysis_path.relative_to(root))

Decisión registrada explícitamente: 128
Resumen descriptivo: results/transformer/token_length_analysis.json


## 4. Configuración experimental

Se estableció el siguiente protocolo para tres experimentos independientes:

- Semilla: 42; duración: 3 épocas.
- Learning rates: `1e-5`, `2e-5` y `3e-5`.
- Weight decay: 0.01.
- Batch físico: 16; acumulación de gradientes: 1; batch efectivo: 16.
- Entrenamiento en un único dispositivo CUDA con precisión fp16.
- Longitud máxima: 128; padding dinámico por batch.
- Optimizador AdamW de PyTorch, scheduler lineal y cero pasos de warmup.
- Evaluación y guardado de checkpoints al finalizar cada época.
- Sin early stopping ni búsqueda automática de hiperparámetros.

La selección se realizó exclusivamente con validation. Se priorizó F1-macro
redondeado a cuatro decimales; los empates se resolvieron por mayor accuracy,
menor learning rate y, finalmente, época más temprana. Test no participó en la
selección de checkpoints ni de hiperparámetros.

## 5. Preparación del entrenamiento

Se prepararon únicamente train y validation para el entrenamiento y la evaluación
de desarrollo. Test se excluyó antes de construir los datasets predictivos.

Se tokenizó `comentario_limpio` con `max_length=128`, `truncation=True`, tokens
especiales y `padding=False`. `DataCollatorWithPadding` aplicó padding dinámico
al formar cada batch.

Los registros del modelo contuvieron únicamente `input_ids`, `attention_mask`,
`token_type_ids` cuando correspondía y `labels`, procedente de `sentimiento_id`.
Los índices originales, el nombre de la partición y el hash del CSV se conservaron
como metadatos separados, sin incorporarlos a los predictores.

Se verificaron la procedencia, el orden de los índices, las etiquetas, los hashes
y la revisión del tokenizer. La interfaz de desarrollo restringió el ajuste a
train y la evaluación a validation; rechazó test y datasets de procedencia distinta.

### Inicialización y checkpoints

Cada experimento partió de BETO original con la revisión registrada, mediante
`AutoModelForSequenceClassification.from_pretrained`. Se establecieron
`num_labels=2`, `id2label={0: "Negativo", 1: "Positivo"}` y el mapeo inverso
`label2id`. La cabeza de clasificación se inicializó para la tarea binaria.
No se reutilizó un modelo afinado de otro experimento.

Se empleó Transformers **4.57.1**, con `eval_strategy="epoch"` y
`processing_class=tokenizer`. En el protocolo oficial,
`load_best_model_at_end=False` y `metric_for_best_model=None` desactivaron la
selección automática de Hugging Face. Se conservaron tres checkpoints, uno por
época, en formato safetensors. La regla de validation se aplicó explícitamente
y el checkpoint seleccionado se cargó por separado.

Las ejecuciones independientes evitaron reanudaciones y sobrescrituras. Los
manifiestos registraron la configuración, los argumentos efectivos, las
versiones del entorno, el modelo original, los hashes y los índices de los datos.

Se calcularon F1-macro, accuracy, precision-macro y recall-macro a partir del
argmax de los logits, con clases fijas `[0, 1]` y `zero_division=0`. También se
contemplaron precisión, recall, F1 y soporte por clase.

### Entorno y reproducibilidad

Se inspeccionaron las versiones de Python, PyTorch y Transformers, el runtime
CUDA y los dispositivos disponibles. La configuración operativa del entrenamiento
se fijó en CUDA, fp16 y un único dispositivo. Se mantuvo la misma configuración
entre los tres experimentos.

Las semillas abarcaron Python, NumPy y PyTorch/CUDA, sin garantizar determinismo
total. El entorno utilizó Transformers 4.57.1 y Accelerate 1.10.1. La carga del
checkpoint original de BETO en formato `.bin` requirió PyTorch >=2.6.
La celda siguiente inspecciona el entorno local sin cargar pesos ni entrenar.

In [ ]:
from scripts.transformer import detect_runtime
runtime_info = detect_runtime()
display(pd.DataFrame([
    {"propiedad": key, "valor": str(value)}
    for key, value in runtime_info.items()
]))
# No construir Trainer ni ejecutar train() en esta fase.

### Comprobación técnica previa

Antes de los experimentos se comprobó el funcionamiento del pipeline en Colab,
con Tesla T4, fp16, batch físico 16 y acumulación 1. Se utilizó un subconjunto
determinista con semilla 42: 32 observaciones de train (16 por clase) y
16 de validation (8 por clase).

La comprobación ejecutó dos pasos de optimización, como máximo una época reducida,
y verificó el cálculo de logits, la propagación de gradientes, la actualización
de la cabeza de clasificación, la evaluación y la serialización del modelo.
La recarga se contrastó sobre un batch de validation.

Estos resultados técnicos se mantuvieron separados de los experimentos y no se
utilizaron para comparar modelos ni seleccionar hiperparámetros. Test no se
tokenizó ni se evaluó durante esta comprobación.

### Separación entre desarrollo y evaluación final

Los tres experimentos utilizaron las mismas particiones y el mismo protocolo,
con learning rates `1e-5`, `2e-5` y `3e-5`. La selección de época y experimento
se completó con validation, aplicando la regla de desempate descrita.

Test quedó reservado para una evaluación predictiva final independiente, con
los modelos y las decisiones metodológicas congelados. Sus métricas no se
utilizan para reajustar modelos, cambiar hiperparámetros ni volver a seleccionar
checkpoints. Las celdas de este notebook no ejecutan entrenamiento ni evaluación
predictiva sobre test.